# Temps de génération par dataset — RTX 4090 Laptop

**Charge unique : 1 000 prix × 2²⁰ = 1 048 576 trajectoires par prix.**  
Pour les formules fermées : 1 000 prix, aucune trajectoire.

Les représentants couvrent transitions exactes, schémas, LSM equity/taux,
FFT et lifts N-facteurs. Chaque mesure utilise les **1 000 lignes originales du catalogue**
(900 core, 100 stress), leurs paramètres et calendriers, pas une tuile répétée.
Ce périmètre ne couvre pas tous les autres produits ou modèles du catalogue.

Ce notebook est **CPU seulement** : il lit les preuves exportées, sans lancer
de GPU, modifier les datasets ni exécuter de validation indépendante.
Une ligne manquante reste manquante, sans extrapolation depuis 64k trajectoires.


In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "tools/performance/export_pricing_dataset_runtime.py").is_file()
)
REPORT_PATH = ROOT / "tests/performance/reports/pricing-dataset-runtime-sm89-2026-09-08.json"
report = json.loads(REPORT_PATH.read_text())
assert report["schema"] == "ai_factory_pricing_dataset_runtime_v1"
assert report["input_profile"] == "ordered_catalogue_900_core_100_stress_v1"
assert (report["price_count"], report["mc_paths_per_price"]) == (1000, 2**20)
cases = report["cases"]
assert cases and len({c["case"] for c in cases}) == len(cases)
assert all(c["rows"] == 1000 and c["paths_per_price"] ==
           (0 if c["family"] == "closed_form" else 2**20) for c in cases)

MODEL_NAMES = {"heston": "Heston", "kou": "Kou", "cir": "CIR",
               "rough_heston": "Rough Heston", "rough_bergomi": "Rough Bergomi",
               "bates": "Bates", "cir_plus_plus": "CIR++",
               "hull_white": "Hull–White", "g2_plus_plus": "G2++",
               "variance_gamma": "Variance Gamma", "normal_inverse_gaussian": "NIG",
               "quadratic_rough_heston": "Quadratic Rough Heston"}
PRODUCT_NAMES = {"mc_terminal": "Européenne call", "mc_barrier": "Barrière down-and-out put",
                 "closed_form": "Caplet — formule fermée", "lsm": "Américaine put — LSM"}
STATUS_NAMES = {"measured": "Mesuré", "running": "En cours", "pending": "À mesurer",
                "failed": "Échec", "inconclusive": "Incomplet", "unavailable": "Indisponible"}

def label(case):
    product = ("Swaption bermudéenne payer — LSM" if case["product"] == "bermudan_swaption"
               and case["family"] == "lsm" else PRODUCT_NAMES[case["family"]])
    model = MODEL_NAMES.get(case["model"], case["model"])
    if case.get("curve"):
        model += " / " + case["curve"].replace("_", " ")
    return model, product

def seconds(value):
    return value / 1000

def duration(value):
    if value is None or pd.isna(value):
        return "—"
    if value < 1:
        return f"{value * 1000:.2f} ms"
    if value < 60:
        return f"{value:.2f} s"
    return f"{int(value // 60)} min {value % 60:04.1f} s"

measured = [c for c in cases if c["status"] == "measured"]
display(Markdown(
    f"**{len(measured)}/{len(cases)} couples mesurés** — export du "
    f"{report['exported_utc']}. "
    + ("Campagne complète." if report["complete"] else
       "**Campagne partielle : aucun total complet ne sera calculé.**")
))


**29/29 couples mesurés** — export du 2026-09-08T21:04:34.188805+00:00. Campagne complète.

## 1. Combien de temps pour 1 000 prix ?

La colonne **génération estimée** additionne des phases mesurées :
préparation initiale + médiane de l'appel hôte synchronisé + copie des résultats
+ une écriture/vérification JSON–YAML locale native. Ce n'est pas un chronomètre
du processus complet à froid. Les warmups et répétitions du benchmark,
l'upload et la certification indépendante ne sont pas facturés à un dataset.

Le temps GPU est affiché séparément : **ne pas l'ajouter au temps hôte**.
Pour MC, il s'agit de l'enveloppe CUDA des appels, pas de la somme des seuls
kernels; pour LSM, de la somme des événements des batches natifs.
L'intervalle affiché varie seulement l'appel hôte entre ses trois mesures :
ce n'est ni une garantie ni un intervalle de confiance.


In [2]:
rows = []
for case in cases:
    model, product = label(case)
    row = {"Modèle": model, "Produit": product,
           "État": STATUS_NAMES.get(case["status"], case["status"]),
           "GPU": "—", "Génération estimée": "—", "Intervalle observé": "—", "CV GPU": "—"}
    if case["timings"] is not None:
        t = case["timings"]
        fixed_ms = t["preparation_ms"] + t["output_copy_ms"] + t["local_publication_ms"]
        host = case["samples"]["raw_host_samples_ms"]
        row.update({
            "GPU": duration(seconds(t["gpu_median_ms"])),
            "Génération estimée": duration(seconds(t["generation_phase_sum_ms"])),
            "Intervalle observé": f"{duration(seconds(fixed_ms + min(host)))} – "
                                  f"{duration(seconds(fixed_ms + max(host)))}",
            "CV GPU": f"{100 * case['gpu_statistics']['coefficient_of_variation']:.1f} %",
        })
    rows.append(row)
display(pd.DataFrame(rows).style.hide(axis="index"))


Modèle,Produit,État,GPU,Génération estimée,Intervalle observé,CV GPU
Bates,Américaine put — LSM,Mesuré,53.82 s,56.85 s,56.09 s – 56.88 s,0.6 %
Bates,Barrière down-and-out put,Mesuré,56.07 s,58.87 s,58.80 s – 58.96 s,0.1 %
Bates,Européenne call,Mesuré,33.56 s,35.27 s,35.16 s – 35.32 s,0.2 %
CIR,Caplet — formule fermée,Mesuré,0.20 ms,277.28 ms,277.28 ms – 277.28 ms,0.0 %
CIR,Swaption bermudéenne payer — LSM,Mesuré,3.02 s,3.81 s,3.73 s – 3.86 s,1.1 %
CIR++ / svensson,Caplet — formule fermée,Mesuré,0.12 ms,299.90 ms,299.90 ms – 299.90 ms,0.2 %
CIR++ / svensson,Swaption bermudéenne payer — LSM,Mesuré,10.43 s,11.63 s,11.54 s – 11.66 s,0.0 %
G2++ / svensson,Caplet — formule fermée,Mesuré,0.01 ms,257.05 ms,257.05 ms – 257.05 ms,27.7 %
G2++ / svensson,Swaption bermudéenne payer — LSM,Mesuré,32.29 s,34.53 s,34.49 s – 34.57 s,0.0 %
Heston,Américaine put — LSM,Mesuré,36.15 s,38.61 s,37.92 s – 38.98 s,1.2 %


In [3]:
if measured:
    # Deux chronomètres différents, volontairement côte à côte, jamais empilés.
    y = np.arange(len(measured))
    gpu = [seconds(c["timings"]["gpu_median_ms"]) for c in measured]
    generation = [seconds(c["timings"]["generation_phase_sum_ms"]) for c in measured]
    fig, ax = plt.subplots(figsize=(11, max(4, 0.62 * len(measured))))
    ax.scatter(gpu, y - 0.12, s=45, color="#20798c", label="Enveloppe GPU", zorder=3)
    ax.scatter(generation, y + 0.12, s=45, color="#d98834",
               label="Génération estimée : somme des phases", zorder=3)
    ax.set_yticks(y, [" · ".join(label(c)) for c in measured])
    ax.invert_yaxis()
    ax.set_xscale("log")
    ax.set_xlabel("Secondes par dataset de 1 000 prix — échelle logarithmique")
    ax.set_title("2²⁰ trajectoires par prix pour MC/LSM ; aucune en formule fermée")
    ax.grid(axis="x", which="major", alpha=0.25)
    ax.set_axisbelow(True)
    ax.legend(loc="best", fontsize=9)
    fig.tight_layout()
    plt.show()
else:
    display(Markdown("Graphique disponible après la première mesure terminée."))


/tmp/ipykernel_312519/2535333686.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. D'où vient le coût ?

La préparation inclut le chargement des inputs existants, les objets préparés
du modèle et les allocations initiales; elle est mesurée une fois par processus.
Ce benchmark ne régénère pas les bases de paramètres elles-mêmes.
Le rough Heston emploie le lift à **sept facteurs**, le rough Bergomi le moteur **FFT**.
Ces temps comparent des couples modèle–produit, pas des modèles à contrat identique :
l'américaine equity, le caplet et la swaption bermudéenne ont des calendriers distincts.

Les warmups sont exclus; leurs nombres et ceux des répétitions sont indiqués ci-dessous.
Les formules fermées regroupent 1 024 appels par échantillon pour éviter de
chronométrer quelques microsecondes : tous ses résultats sont ramenés à **un**
dataset de 1 000 prix. Les horloges hôte et CUDA sont conservées séparément,
même si elles divergent; aucune valeur brute n'est artificiellement remontée.


In [4]:
phase_rows = []
for c in measured:
    t = c["timings"]
    phase_rows.append({
        "Couple": " · ".join(label(c)),
        "Préparation": duration(seconds(t["preparation_ms"])),
        "Appel hôte": duration(seconds(t["raw_host_api_ms"])),
        "Copie D2H": duration(seconds(t["output_copy_ms"])),
        "Publication locale": duration(seconds(t["local_publication_ms"])),
        "CV hôte": f"{100 * c['raw_host_statistics']['coefficient_of_variation']:.1f} %",
        "Appels / échantillon": c["operations_per_sample"],
        "Warmups / mesures": f"{c['configuration']['warmups']} / {c['configuration']['repetitions']}",
    })
if phase_rows:
    display(pd.DataFrame(phase_rows).style.hide(axis="index"))

clock_gaps = [100 * (c["timings"]["gpu_median_ms"] / c["timings"]["raw_host_api_ms"] - 1)
              for c in measured if c["timings"]["raw_host_api_ms"] > 0]
if clock_gaps and max(clock_gaps) > 1:
    display(Markdown(
        f"**Écart entre chronomètres :** GPU / hôte − 1 va de {min(clock_gaps):.1f} % "
        f"à {max(clock_gaps):.1f} % sur ces couples. Des durées CUDA dépassent donc "
        "l'appel hôte synchronisé; l'origine de cet écart n'est pas établie ici. "
        "Les estimations utilisent l'horloge hôte, sans correction artificielle, "
        "et ne démontrent pas une précision absolue de 1 %."
    ))

warnings = []
for c in cases:
    name = " · ".join(label(c))
    if c["status"] != "measured":
        warnings.append(f"**{name}** : {STATUS_NAMES.get(c['status'], c['status'])}"
                        + (f" — {c['reason']}" if c.get("reason") else "") + ".")
        continue
    if not c["environment_eligible_for_tuning"]:
        warnings.append(f"**{name}** : environnement non qualifiant pour un retuning"
                        f" ({'; '.join(c['timing_ineligibility_reasons']) or 'voir le brut'}).")
    if c["gpu_statistics"]["coefficient_of_variation"] > .05:
        warnings.append(f"**{name}** : CV GPU > 5 %, estimation plus variable; "
                        "pas de configuration qualifiée à partir de ce point.")
display(Markdown("**Réserves de mesure**\n\n" +
                 ("\n".join("- " + warning for warning in warnings)
                  if warnings else "Aucune alerte d'environnement ou de CV GPU > 5 % sur ces points.")))


Couple,Préparation,Appel hôte,Copie D2H,Publication locale,CV hôte,Appels / échantillon,Warmups / mesures
Bates · Américaine put — LSM,195.26 ms,56.64 s,0.14 ms,14.65 ms,0.7 %,1,2 / 3
Bates · Barrière down-and-out put,267.34 ms,58.59 s,0.21 ms,13.22 ms,0.1 %,1,2 / 3
Bates · Européenne call,213.19 ms,35.05 s,0.26 ms,12.69 ms,0.2 %,1,2 / 3
CIR · Caplet — formule fermée,266.24 ms,0.21 ms,0.16 ms,10.66 ms,0.0 %,1024,2 / 3
CIR · Swaption bermudéenne payer — LSM,209.89 ms,3.58 s,0.11 ms,12.18 ms,1.4 %,1,2 / 3
CIR++ / svensson · Caplet — formule fermée,284.68 ms,0.12 ms,0.16 ms,14.93 ms,0.2 %,1024,2 / 3
CIR++ / svensson · Swaption bermudéenne payer — LSM,194.76 ms,11.41 s,0.14 ms,17.68 ms,0.4 %,1,2 / 3
G2++ / svensson · Caplet — formule fermée,239.45 ms,0.01 ms,0.09 ms,17.49 ms,27.4 %,1024,2 / 3
G2++ / svensson · Swaption bermudéenne payer — LSM,253.32 ms,34.26 s,0.18 ms,20.50 ms,0.1 %,1,2 / 3
Heston · Américaine put — LSM,232.85 ms,38.36 s,0.17 ms,14.07 ms,1.1 %,1,2 / 3


**Réserves de mesure**

- **CIR++ / svensson · Swaption bermudéenne payer — LSM** : environnement non qualifiant pour un retuning (GPU power limit changed by more than 10%).
- **G2++ / svensson · Caplet — formule fermée** : CV GPU > 5 %, estimation plus variable; pas de configuration qualifiée à partir de ce point.
- **Hull–White / svensson · Caplet — formule fermée** : CV GPU > 5 %, estimation plus variable; pas de configuration qualifiée à partir de ce point.
- **Kou · Européenne call** : CV GPU > 5 %, estimation plus variable; pas de configuration qualifiée à partir de ce point.

## 3. Géométries et mémoire

Les réglages sont les références actuelles des recettes, **pas un nouvel optimum
accepté**, ni une configuration universelle. Le FFT garde ses kernels internes
spécialisés, traite les prix successivement et découpe leurs trajectoires.
Le LSM garde son planner de batches selon la VRAM; le nombre de batches ci-dessous
est celui réellement retourné par le launcher.
« Blocs lancés max » est le maximum d'une grille de kernel, pas la somme de
tous les lancements. Les threads demandés concernent le profil principal;
les kernels auxiliaires peuvent avoir leur propre géométrie, conservée dans le JSON.

« Mémoire suivie » est le pic comptabilisé par l'outillage, pas la consommation
totale du processus. La mémoire locale compilée n'est pas à elle seule une
mesure des spills à l'exécution.


In [5]:
resource_rows = []
for c in measured:
    cfg, mem = c["configuration"], c["memory"]
    resources = c["resources"]
    batches = c.get("lsm_batches") or []
    resource_rows.append({
        "Couple": " · ".join(label(c)),
        "Threads / bloc demandés": cfg["requested_threads"] or "FFT spécialisé",
        "Blocs lancés max": max((r.get("grid_blocks_max", 0) for r in resources), default=0),
        "Blocs / prix LSM": cfg["blocks_per_price"] or "—",
        "Prix / appel hôte": cfg["batch_rows"],
        "Chunk FFT": cfg["path_chunk"] or "—",
        "Batches VRAM LSM": sum(b["native_batch_count"] for b in batches) if batches else "—",
        "Mémoire suivie": (f"{mem['tracked_peak_bytes'] / 2**20:.1f} Mio"
                            if mem['tracked_peak_bytes'] < 2**30
                            else f"{mem['tracked_peak_bytes'] / 2**30:.2f} Gio"),
        "Registres max / thread": max((r["registers"] for r in resources), default=0),
        "Local max (octets / thread)": max((r["local_bytes"] for r in resources), default=0),
    })
if resource_rows:
    display(pd.DataFrame(resource_rows).style.hide(axis="index"))
if measured:
    env = measured[0]["environment"]
    display(Markdown(
        f"Machine : **{env['gpu']}**, SM{env['compute_capability'].replace('.', '')}, "
        f"{env['sm_count']} SM, {env['memory_bytes'] / 2**30:.2f} Gio. "
        f"CUDA compilateur {env['cuda_compiler_version']}; versions API runtime/driver "
        f"{env['runtime_version']}/{env['driver_version']}. "
        "Télémétrie conservée, sans seuil d'arrêt thermique applicatif ni modification matérielle."
    ))


Couple,Threads / bloc demandés,Blocs lancés max,Blocs / prix LSM,Prix / appel hôte,Chunk FFT,Batches VRAM LSM,Mémoire suivie,Registres max / thread,Local max (octets / thread)
Bates · Américaine put — LSM,128,21760,128,1000,—,32,13.22 Gio,88,0
Bates · Barrière down-and-out put,512,1000,—,1000,—,—,0.1 Mio,80,0
Bates · Européenne call,512,1000,—,1000,—,—,0.1 Mio,76,0
CIR · Caplet — formule fermée,256,4,—,1000,—,—,0.0 Mio,40,0
CIR · Swaption bermudéenne payer — LSM,128,31680,64,1000,—,3,13.19 Gio,79,0
CIR++ / svensson · Caplet — formule fermée,256,4,—,1000,—,—,0.1 Mio,43,0
CIR++ / svensson · Swaption bermudéenne payer — LSM,128,31680,64,1000,—,3,13.19 Gio,108,0
G2++ / svensson · Caplet — formule fermée,256,4,—,1000,—,—,0.1 Mio,39,0
G2++ / svensson · Swaption bermudéenne payer — LSM,128,11392,64,1000,—,6,13.12 Gio,148,0
Heston · Américaine put — LSM,128,21760,128,1000,—,32,13.22 Gio,88,0


Machine : **NVIDIA GeForce RTX 4090 Laptop GPU**, SM89, 76 SM, 15.99 Gio. CUDA compilateur 13.3.73; versions API runtime/driver 13030/13020. Télémétrie conservée, sans seuil d'arrêt thermique applicatif ni modification matérielle.

## 4. Budget de la campagne prévue

Modifier seulement les nombres ci-dessous : **1** signifie un dataset de
1 000 prix pour ce couple, **0** l'exclut. Le total est une somme séquentielle
des estimations, pas une mesure de lancement simultané ou du catalogue entier.
Les régimes de fréquence et de puissance peuvent évoluer pendant une longue
campagne. Un autre GPU doit refaire les mesures.

Multiplier ce budget par 1 000 ne certifierait pas le temps de 1 million de
prix : cette campagne mesure un seul point de charge, pas la linéarité en prix
ou trajectoires. Les comparaisons historiques restent dans le
[rapport de scaling](pricing-workload-scaling-sm89-2026-09-07.md).


In [6]:
# Nombre de datasets de 1 000 prix à générer pour chaque couple mesuré.
DATASETS_PER_CASE = {case["case"]: 1 for case in cases}

by_id = {c["case"]: c for c in cases}
assert set(DATASETS_PER_CASE) <= set(by_id), "Couple non mesuré dans cette campagne"
assert all(type(n) is int and n >= 0 for n in DATASETS_PER_CASE.values())
selected = [(by_id[key], count) for key, count in DATASETS_PER_CASE.items() if count]
missing = [c["case"] for c, _ in selected if c["timings"] is None]
if missing:
    display(Markdown("**Total indisponible** : mesures manquantes pour " + ", ".join(missing) + "."))
elif not selected:
    display(Markdown("Aucun dataset sélectionné."))
else:
    total_ms = sum(c["timings"]["generation_phase_sum_ms"] * n for c, n in selected)
    dataset_count = sum(n for _, n in selected)
    price_total = f"{1000 * dataset_count:,}".replace(",", " ")
    display(Markdown(
        f"**{dataset_count} datasets, soit {price_total} prix : "
        f"{duration(seconds(total_ms))} estimées en séquentiel.**\n\n"
        "Ce total comprend la préparation de chaque dataset. Il exclut la génération des bases "
        "de paramètres, l'upload et la validation indépendante; il n'est pas un délai garanti."
    ))


**29 datasets, soit 29 000 prix : 16 min 57.0 s estimées en séquentiel.**

Ce total comprend la préparation de chaque dataset. Il exclut la génération des bases de paramètres, l'upload et la validation indépendante; il n'est pas un délai garanti.

## 5. Ce que ces résultats valident — et ne valident pas

Les sorties sont finies et reproduites à géométrie identique. Cela ne constitue
**pas une certification indépendante des prix**. La sensibilité Kou a été corrigée
sous **NUM-017**, avec référence de résolution haute précision, plusieurs géométries
et coût explicite; voir le [rapport de préparation](catalogue-generation-readiness-sm89-2026-09-08.md).
Le seul succès d'un replay ne suffirait pas à cette conclusion.

Les trois constats de scaling **PERF-016, PERF-017, PERF-019** conservent leurs
critères multi-tailles, ressources et qualification. Cette campagne ne les clôt
pas par analogie et ne qualifie ni les autres modèles/produits, ni les samples,
ni d'autres GPU.

### Provenance et actualisation

Le JSON portable conserve les échantillons, ressources, configuration et SHA-256
du snapshot, des binaires, des inputs et des preuves brutes. Les résultats de
publication du benchmark restent temporaires sous `build-dev`; les datasets du
catalogue ne sont pas écrasés. La révision seule ne décrit pas le worktree modifié :
les empreintes du diff et de l'archive source sont donc indispensables.

Pour actualiser **sans relancer le GPU**, exécuter depuis la racine :
```bash
python tools/performance/export_pricing_dataset_runtime.py \
  build-dev/pricing-dataset-runtime-20260908-01 \
  --output tests/performance/reports/pricing-dataset-runtime-sm89-2026-09-08.json
```
Puis exécuter les cellules du notebook. Pour refaire la mesure, utiliser le
[protocole de performance](../performance-regression-protocol.md) et le
`plan.json` archivé de la campagne, dont les géométries sont aussi dans le JSON
portable. Ne pas mélanger cette campagne avec le profil historique de tuile répétée.


In [7]:
display(pd.DataFrame([
    {"Preuve": "Révision", "Valeur": report["revision"]},
    {"Preuve": "Diff source SHA-256", "Valeur": report["source_diff_sha256"]},
    {"Preuve": "Archive source SHA-256", "Valeur": report["source_archive_sha256"]},
    {"Preuve": "Campagne brute", "Valeur": report["campaign"]},
    {"Preuve": "Début UTC", "Valeur": report["started_utc"]},
    {"Preuve": "Export portable", "Valeur": str(REPORT_PATH.relative_to(ROOT))},
]).style.hide(axis="index"))

# Échantillons bruts normalisés : chaque ligne reste un dataset de 1 000 prix.
display(pd.DataFrame([
    {"Couple": " · ".join(label(c)),
     "GPU (s)": [round(v / 1000, 6) for v in c["samples"]["gpu_samples_ms"]],
     "Hôte (s)": [round(v / 1000, 6) for v in c["samples"]["raw_host_samples_ms"]]}
    for c in measured
]).style.hide(axis="index"))


Preuve,Valeur
Révision,872a986b1f0947a1a832af0615ffc6d80dbedb81
Diff source SHA-256,8db431ffea77d003ae6c7aa353eea1191a5e99423b6e27dd3d2049c869a01172
Archive source SHA-256,e0a954ef5a8ca1a0ac1663ca46153b7de3a80bdd78e4dc03d1ea9e10c9482703
Campagne brute,build-dev/pricing-dataset-runtime-20260908-01
Début UTC,2026-09-08T19:55:46.930079+00:00
Export portable,tests/performance/reports/pricing-dataset-runtime-sm89-2026-09-08.json


Couple,GPU (s),Hôte (s)
Bates · Américaine put — LSM,"[53.106655, 53.832999, 53.818239]","[55.876567, 56.638931, 56.669636]"
Bates · Barrière down-and-out put,"[56.178289, 56.030176, 56.065559]","[58.682266, 58.515163, 58.585365]"
Bates · Européenne call,"[33.432859, 33.561477, 33.557437]","[34.937912, 35.090338, 35.045129]"
CIR · Caplet — formule fermée,"[0.000202, 0.000202, 0.000202]","[0.00021, 0.00021, 0.00021]"
CIR · Swaption bermudéenne payer — LSM,"[2.954138, 3.024369, 3.024081]","[3.512584, 3.584111, 3.634714]"
CIR++ / svensson · Caplet — formule fermée,"[0.000117, 0.000116, 0.000116]","[0.000122, 0.000122, 0.000122]"
CIR++ / svensson · Swaption bermudéenne payer — LSM,"[10.421226, 10.427037, 10.433856]","[11.326985, 11.412558, 11.448969]"
G2++ / svensson · Caplet — formule fermée,"[7e-06, 1.4e-05, 1.3e-05]","[7e-06, 1.5e-05, 1.4e-05]"
G2++ / svensson · Swaption bermudéenne payer — LSM,"[32.275124, 32.285598, 32.288456]","[34.21101, 34.259459, 34.297046]"
Heston · Américaine put — LSM,"[35.520045, 36.147334, 36.57555]","[37.676671, 38.358985, 38.73287]"
